In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Predicting Bus Fuel Consumption") \
    .getOrCreate()

print(spark.version)

4.1.2


In [2]:
df = spark.read.csv(
    r"C:\Users\ASUS\Documents\Bus project\data\bus_fuel_consumption_114k.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+----------------+------------+---------+--------+---------------+--------------+---------+--------+-------+-------------+-------------+-------------+----------------+----------------+
|        Operator|Service_Code|Line_Name|Distance|Number_of_Stops|Departure_Hour|Peak_Hour|Day_Type|Weather|Traffic_Level|Delay_Minutes|Average_Speed|Fuel_Consumption|Efficiency_Class|
+----------------+------------+---------+--------+---------------+--------------+---------+--------+-------+-------------+-------------+-------------+----------------+----------------+
|Stagecoach South|PK0002571:21|       46|     437|            264|             7|        1| Weekday|   Rain|         High|           17|           40|          210.65|             Low|
|Stagecoach South|PK0002571:21|       46|     437|            264|             7|        1| Weekday|   Rain|         High|           13|           39|          209.38|          Medium|
|Stagecoach South|PK0002571:21|       46|     437|            264|         

In [3]:
df.printSchema()

root
 |-- Operator: string (nullable = true)
 |-- Service_Code: string (nullable = true)
 |-- Line_Name: string (nullable = true)
 |-- Distance: integer (nullable = true)
 |-- Number_of_Stops: integer (nullable = true)
 |-- Departure_Hour: integer (nullable = true)
 |-- Peak_Hour: integer (nullable = true)
 |-- Day_Type: string (nullable = true)
 |-- Weather: string (nullable = true)
 |-- Traffic_Level: string (nullable = true)
 |-- Delay_Minutes: integer (nullable = true)
 |-- Average_Speed: integer (nullable = true)
 |-- Fuel_Consumption: double (nullable = true)
 |-- Efficiency_Class: string (nullable = true)



In [4]:
print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 114000
Columns: 14


In [5]:
from pyspark.sql.functions import col, when, count

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+--------+------------+---------+--------+---------------+--------------+---------+--------+-------+-------------+-------------+-------------+----------------+----------------+
|Operator|Service_Code|Line_Name|Distance|Number_of_Stops|Departure_Hour|Peak_Hour|Day_Type|Weather|Traffic_Level|Delay_Minutes|Average_Speed|Fuel_Consumption|Efficiency_Class|
+--------+------------+---------+--------+---------------+--------------+---------+--------+-------+-------------+-------------+-------------+----------------+----------------+
|       0|           0|        0|       0|              0|             0|        0|       0|      0|            0|            0|            0|               0|               0|
+--------+------------+---------+--------+---------------+--------------+---------+--------+-------+-------------+-------------+-------------+----------------+----------------+



In [6]:
print("Total Records :", df.count())
print("Unique Records:", df.dropDuplicates().count())

Total Records : 114000
Unique Records: 113936


In [7]:
df = df.dropDuplicates()

print("Rows after removing duplicates:", df.count())

Rows after removing duplicates: 113936


In [8]:
df.describe().show()

+-------+----------------+-------------+------------------+------------------+------------------+------------------+------------------+--------+-------+-------------+------------------+-----------------+------------------+----------------+
|summary|        Operator| Service_Code|         Line_Name|          Distance|   Number_of_Stops|    Departure_Hour|         Peak_Hour|Day_Type|Weather|Traffic_Level|     Delay_Minutes|    Average_Speed|  Fuel_Consumption|Efficiency_Class|
+-------+----------------+-------------+------------------+------------------+------------------+------------------+------------------+--------+-------+-------------+------------------+-----------------+------------------+----------------+
|  count|          113936|       113936|            113936|            113936|            113936|            113936|            113936|  113936| 113936|       113936|            113936|           113936|            113936|          113936|
|   mean|            NULL|         NULL|

In [9]:
from pyspark.sql.functions import when

df = df.withColumn(
    "Peak_Hour",
    when(
        (col("Departure_Hour") >= 7) &
        (col("Departure_Hour") <= 9),
        1
    ).when(
        (col("Departure_Hour") >= 16) &
        (col("Departure_Hour") <= 18),
        1
    ).otherwise(0)
)

df.select(
    "Departure_Hour",
    "Peak_Hour"
).show(20)

+--------------+---------+
|Departure_Hour|Peak_Hour|
+--------------+---------+
|             7|        1|
|             7|        1|
|             7|        1|
|             7|        1|
|             7|        1|
|             7|        1|
|             7|        1|
|             8|        1|
|             8|        1|
|             8|        1|
|             8|        1|
|             6|        0|
|             6|        0|
|             6|        0|
|             7|        1|
|             7|        1|
|             7|        1|
|             7|        1|
|             7|        1|
|             7|        1|
+--------------+---------+
only showing top 20 rows


In [10]:
from pyspark.sql.functions import when

df = df.withColumn(
    "Efficiency_Class",
    when(col("Number_of_Stops") >= 2000, "High")
    .when(col("Number_of_Stops") >= 800, "Medium")
    .otherwise("Low")
)

df.groupBy("Efficiency_Class").count().show()

+----------------+-----+
|Efficiency_Class|count|
+----------------+-----+
|            High|32485|
|             Low|43981|
|          Medium|37470|
+----------------+-----+



In [11]:
from pyspark.ml.feature import StringIndexer

label_indexer = StringIndexer(
    inputCol="Efficiency_Class",
    outputCol="label"
)

indexer_model = label_indexer.fit(df)
df = indexer_model.transform(df)

df.select("Efficiency_Class", "label").show(10)


+----------------+-----+
|Efficiency_Class|label|
+----------------+-----+
|             Low|  0.0|
|             Low|  0.0|
|            High|  2.0|
|            High|  2.0|
|            High|  2.0|
|            High|  2.0|
|            High|  2.0|
|            High|  2.0|
|            High|  2.0|
|            High|  2.0|
+----------------+-----+
only showing top 10 rows


In [12]:
from pyspark.ml.feature import VectorAssembler

feature_columns = [
    "Distance",
    "Number_of_Stops",
    "Departure_Hour",
    "Peak_Hour"
]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

df = assembler.transform(df)

df.select("features", "label").show(5, truncate=False)

+----------------------+-----+
|features              |label|
+----------------------+-----+
|[437.0,264.0,7.0,1.0] |0.0  |
|[437.0,109.0,7.0,1.0] |0.0  |
|[721.0,3854.0,7.0,1.0]|2.0  |
|[721.0,3854.0,7.0,1.0]|2.0  |
|[721.0,3854.0,7.0,1.0]|2.0  |
+----------------------+-----+
only showing top 5 rows


In [13]:
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", train_data.count())
print("Testing rows:", test_data.count())

Training rows: 91335
Testing rows: 22601


In [14]:
from pyspark.sql.functions import when, col

df = df.withColumn(
    "Peak_Hour",
    when(
        ((col("Departure_Hour") >= 7) & (col("Departure_Hour") <= 9)) |
        ((col("Departure_Hour") >= 16) & (col("Departure_Hour") <= 18)),
        1
    ).otherwise(0)
)


In [15]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Train model
lr = LogisticRegression(featuresCol="features", labelCol="label")

lr_model = lr.fit(train_data)

# Prediction
lr_predictions = lr_model.transform(test_data)

# Show predictions
lr_predictions.select(
    "Efficiency_Class",
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

+----------------+-----+----------+---------------------------------------------------+
|Efficiency_Class|label|prediction|probability                                        |
+----------------+-----+----------+---------------------------------------------------+
|Low             |0.0  |0.0       |[1.0,3.993244806516788E-40,7.8868129659135675E-199]|
|Low             |0.0  |0.0       |[1.0,3.993244806516788E-40,7.8868129659135675E-199]|
|Low             |0.0  |0.0       |[1.0,3.993244806516788E-40,7.8868129659135675E-199]|
|Low             |0.0  |0.0       |[1.0,3.993244806516788E-40,7.8868129659135675E-199]|
|Low             |0.0  |0.0       |[1.0,3.993244806516788E-40,7.8868129659135675E-199]|
|Low             |0.0  |0.0       |[1.0,3.993244806516788E-40,7.8868129659135675E-199]|
|Low             |0.0  |0.0       |[1.0,3.993244806516788E-40,7.8868129659135675E-199]|
|Low             |0.0  |0.0       |[1.0,3.993244806516788E-40,7.8868129659135675E-199]|
|Low             |0.0  |0.0     

In [16]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

lr_accuracy = evaluator.evaluate(lr_predictions)

print("Logistic Regression Accuracy:", lr_accuracy)

Logistic Regression Accuracy: 1.0


In [17]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    maxDepth=5,
    seed=42
)

dt_model = dt.fit(train_data)

dt_predictions = dt_model.transform(test_data)

dt_accuracy = evaluator.evaluate(dt_predictions)

print("Decision Tree Accuracy:", dt_accuracy)

Decision Tree Accuracy: 1.0


In [18]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    seed=42
)

rf_model = rf.fit(train_data)

rf_predictions = rf_model.transform(test_data)

rf_accuracy = evaluator.evaluate(rf_predictions)

print("Random Forest Accuracy:", rf_accuracy)

Random Forest Accuracy: 0.9957966461660989


In [19]:
print("=" * 40)
print("Model Comparison")
print("=" * 40)

print("Logistic Regression :", round(lr_accuracy, 4))
print("Decision Tree       :", round(dt_accuracy, 4))
print("Random Forest       :", round(rf_accuracy, 4))

Model Comparison
Logistic Regression : 1.0
Decision Tree       : 1.0
Random Forest       : 0.9958


In [20]:
rf_predictions.groupBy("label", "prediction").count().show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  1.0|       1.0| 7507|
|  0.0|       1.0|   95|
|  2.0|       2.0| 6399|
|  0.0|       0.0| 8600|
+-----+----------+-----+



In [21]:
feature_columns = [
    "Distance",
    "Departure_Hour",
    "Peak_Hour"
]

In [22]:
df = df.drop("features")

In [23]:
from pyspark.ml.feature import VectorAssembler

feature_columns = [
    "Distance",
    "Departure_Hour",
    "Peak_Hour"
]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

df = assembler.transform(df)

df.select("features", "label").show(5, truncate=False)

+---------------+-----+
|features       |label|
+---------------+-----+
|[437.0,7.0,1.0]|0.0  |
|[437.0,7.0,1.0]|0.0  |
|[721.0,7.0,1.0]|2.0  |
|[721.0,7.0,1.0]|2.0  |
|[721.0,7.0,1.0]|2.0  |
+---------------+-----+
only showing top 5 rows


In [24]:
print(df.columns)


['Operator', 'Service_Code', 'Line_Name', 'Distance', 'Number_of_Stops', 'Departure_Hour', 'Peak_Hour', 'Day_Type', 'Weather', 'Traffic_Level', 'Delay_Minutes', 'Average_Speed', 'Fuel_Consumption', 'Efficiency_Class', 'label', 'features']


In [25]:
df = df.drop("features")

In [26]:
from pyspark.ml.feature import VectorAssembler

feature_columns = [
    "Distance",
    "Departure_Hour",
    "Peak_Hour"
]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

df = assembler.transform(df)

df.select("features", "label").show(5, truncate=False)

+---------------+-----+
|features       |label|
+---------------+-----+
|[437.0,7.0,1.0]|0.0  |
|[437.0,7.0,1.0]|0.0  |
|[721.0,7.0,1.0]|2.0  |
|[721.0,7.0,1.0]|2.0  |
|[721.0,7.0,1.0]|2.0  |
+---------------+-----+
only showing top 5 rows
